<a href="https://colab.research.google.com/github/KEERTHANASRI-A-M-0609/keerthanasri-codeboosters-2026/blob/main/Day%204/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The 5 Vs of Big Data

#1.Volume
#2.Velocity
#3.Variety
#4.Veracity
#5.Value

In [4]:
!pip install pyspark --quiet

print('Pyspark installation Complete!')

Pyspark installation Complete!


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month, to_date, col, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#Create SparkSession

spark = SparkSession.builder \
    .appName('Day4_Bigdata_Sales') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()


print(f'Spark Version : {spark.version}')
print(f'SparkSession : ACTIVE')
print(f'Application : {spark.sparkContext.appName}')


Spark Version : 4.0.2
SparkSession : ACTIVE
Application : Day4_Bigdata_Sales


In [6]:
df_bronze = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv('large_sales_data.csv')
print('=== BRONZE LAYER - Raw Data ===')
print(f'Rows : {df_bronze.count()}')
print(f'Columns : {len(df_bronze.columns)}')
print(f'Names : {df_bronze.columns}')
print()
df_bronze.printSchema()

=== BRONZE LAYER - Raw Data ===
Rows : 5000
Columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [7]:
print('First 5 rows: ')
df_bronze.show(5, truncate=False)
print('\n')
print(df_bronze.tail(5))
#print(pd.DataFrame(df_bronze.tail(5)))  #--for tables form

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity', 'unit_price', 'revenue').describe().show()

First 5 rows: 
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi K

In [8]:
df_bronze.write \
    .mode('overwrite') \
    .parquet('sales_bronze.parquet')

In [9]:
import os

def get_dir_size(path):
    """Get total size of a file or directory in KB."""
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024
    total = 0

    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1024

csv_size = get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1 - (parquet_size / csv_size)) * 100
print(f'CSV size: {csv_size:.1f} KB')
print(f'Parquet size: {parquet_size:.1f} KB')
print(f'Reduction: {reduction:.1f}% smaller')
print(f'\nAt 1 TB scale: CSV=1000 GB -> Parquet={1000*(1-reduction/100):.0f} GB')

CSV size: 529.3 KB
Parquet size: 55.1 KB
Reduction: 89.6% smaller

At 1 TB scale: CSV=1000 GB -> Parquet=104 GB


In [10]:
print('\nBasic statistics for numeric columns:')
df_bronze = df_bronze.withColumn("total_price_revenue", F.col("unit_price") + F.col("revenue"))
df_bronze.select('quantity', 'unit_price', 'revenue','total_price_revenue').describe().show()


Basic statistics for numeric columns:
+-------+-----------------+------------------+------------------+-------------------+
|summary|         quantity|        unit_price|           revenue|total_price_revenue|
+-------+-----------------+------------------+------------------+-------------------+
|  count|             5000|              5000|              5000|               5000|
|   mean|           7.9536|          12496.86|          99169.52|          111666.38|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103| 158330.85825520795|
|    min|                1|               600|               600|               1200|
|    max|               15|             45000|            675000|             720000|
+-------+-----------------+------------------+------------------+-------------------+



In [11]:
df_bronze.select("product","revenue","city").show(5)

+----------+-------+---------+
|   product|revenue|     city|
+----------+-------+---------+
|   Monitor| 264000|   Mumbai|
|   Printer| 120000|    Delhi|
|     Mouse|   8000|Ahmedabad|
|    Tablet| 160000|    Surat|
|Headphones|  14000|Bangalore|
+----------+-------+---------+
only showing top 5 rows


In [12]:
#FILTER
df_bronze.filter(F.col("revenue") > 50000).show()
#OR df_bronze.filter(df.revenue > 50000)
#OR df_bronze.where("revenue" > 50000)

+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-------------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|total_price_revenue|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-------------------+
|    1001|  Sneha Reddy|Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Delivered|             286000|
|    1002| Ramesh Kumar|Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North| Anil Sharma|     Credit Card|     Shipped|             132000|
|    1004|   Suresh Rao| Tablet|Electronics|       5|     32000| 160000|2023-01-04|    Surat|  West|  Ravi Kumar|Cash on Delivery|  Processing|             192000|
|    1008|  Priy

In [13]:
df_bronze.groupBy("category") \
         .agg(F.sum("revenue").alias("total_revenue"),
              F.count("*").alias("orders"),
              F.avg("revenue").alias("avg_revenue")) \
         .orderBy("total_revenue", ascending=False)

DataFrame[category: string, total_revenue: bigint, orders: bigint, avg_revenue: double]

 DATA STORAGE: WAREHOUSE -> LAKE -> LAKEHOUSE

In [14]:
df_silver = df_bronze \
    .dropDuplicates() \
    .dropna(subset = ['order_id', 'product','revenue'])

df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

df_silver = df_silver \
    .withColumn('year', year(col('order_date'))) \
    .withColumn('month', month(col('order_date'))) \

df_silver = df_silver.withColumn(
    'revenue_category',
  F.when(col('revenue') > 40000, 'High')
   .when(col('revenue') > 10000, 'Medium')
   .otherwise('Low')
)

print(f'Silver layer rows: {df_silver.count()}')
print('New columns added: order_year, order_month', 'revenue_category')
df_silver.select('product', 'revenue', 'year', 'month', 'revenue_category').show(8)

Silver layer rows: 5000
New columns added: order_year, order_month revenue_category
+-------+-------+----+-----+----------------+
|product|revenue|year|month|revenue_category|
+-------+-------+----+-----+----------------+
|Monitor| 242000|2023|   11|            High|
| Webcam|  27500|2023|    1|          Medium|
|Printer| 156000|2023|    9|            High|
|Printer|  84000|2023|    8|            High|
|Monitor| 110000|2023|   11|            High|
|Printer| 132000|2023|    6|            High|
|Printer| 168000|2023|    4|            High|
|  Mouse|   9600|2023|    2|             Low|
+-------+-------+----+-----+----------------+
only showing top 8 rows


In [15]:
df_dropped_duplicates = df_bronze.join(df_silver, on=['order_id'], how='left_anti')

print(f"Total duplicate rows dropped: {df_dropped_duplicates.count()}")
print("Sample of dropped duplicate rows:")
df_dropped_duplicates.select('order_id', 'customer_name', 'product', 'revenue').show(8, truncate=False)

Total duplicate rows dropped: 0
Sample of dropped duplicate rows:
+--------+-------------+-------+-------+
|order_id|customer_name|product|revenue|
+--------+-------------+-------+-------+
+--------+-------------+-------+-------+



In [16]:
df_silver.write \
    .mode('overwrite') \
    .parquet('sales_silver.parquet')

print('Silver Parquet saved: sales_silver.parquet')
print(f'Silver Parquet size: {get_dir_size("sales_silver.parquet"):.1f} KB')

df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Read-back rows: {df_verify.count()} (should match Silver count)')
df_verify.printSchema()

Silver Parquet saved: sales_silver.parquet
Silver Parquet size: 64.9 KB
Read-back rows: 5000 (should match Silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total_price_revenue: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [17]:
df_bronze.write \
    .mode('overwrite') \
    .parquet('sales_bronze.parquet')

print('Bronze Parquet saved: sales_bronze.parquet')
print(f'Bronze Parquet size: {get_dir_size("sales_bronze.parquet"):.1f} KB')

df_verify_bronze = spark.read.parquet('sales_bronze.parquet')
print(f'Read-back rows: {df_verify_bronze.count()} (should match Bronze count)')
df_verify_bronze.printSchema()

Bronze Parquet saved: sales_bronze.parquet
Bronze Parquet size: 60.2 KB
Read-back rows: 5000 (should match Bronze count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total_price_revenue: integer (nullable = true)



In [19]:
top_products = df_silver \
    .groupBy('product') \
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('order_id').alias('num_orders'),
        F.avg('revenue').alias('avg_order_revenue')
    ) \
    .orderBy('total_revenue', ascending = False) \
    .limit(5)

print('=== Top 5 Products by REvenue ===')
top_products.show(truncate = False)

=== Top 5 Products by REvenue ===
+-------+-------------+----------+------------------+
|product|total_revenue|num_orders|avg_order_revenue |
+-------+-------------+----------+------------------+
|Laptop |182700000    |502       |363944.22310756973|
|Tablet |135104000    |532       |253954.8872180451 |
|Monitor|82126000     |481       |170740.12474012474|
|Printer|44544000     |488       |91278.68852459016 |
|Speaker|16317000     |470       |34717.02127659575 |
+-------+-------------+----------+------------------+

